# Benchmark matériel du modèle `EEGVAE`

Mesure des performances **matérielles** (et non de la qualité de reconstruction) :
taille du modèle, FLOPs/MACs, latence, throughput, mémoire GPU et scaling.

> ⚠️ Le modèle utilise des blocs **Mamba** (scan sélectif, souvent kernel CUDA custom).
> Les compteurs de FLOPs classiques ne connaissent pas ces opérateurs et vont
> **sous-compter** : traite toute valeur de FLOPs comme une **borne inférieure**
> et regarde toujours la liste des `unsupported ops`.

**Dépendances optionnelles** (le notebook fonctionne sans, en dégradé) :
```bash
pip install fvcore calflops
```

## 1. Setup, configuration et construction du modèle

In [1]:
import time
import numpy as np
import torch

from model.model import EEGVAE

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)
if device == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# ---- Paramètres du benchmark ----
B, C, T = 8, 64, 6000       # batch, canaux EEG, longueur temporelle (échantillons)
FS = 250                    # fréquence d'échantillonnage (Hz) -> pour le débit en s-EEG/s
DTYPE = torch.float32       # passe en torch.float16 / torch.bfloat16 pour tester l'AMP

# ---- Configuration du modèle (adapte-la à ta config réelle) ----
config = dict(
    in_channels=C,
    out_channels=C,
    z_channels=16,
    ch=64,
    ch_mult=[1, 2, 4],
    n_blocks=2,
    dropout=0.0,
    mode="vq",              # "vq" ou "kl"
    num_embeddings=512,
    n_quantizers=4,
    commitment_cost=0.25,
    decay=0.99,
)

model = EEGVAE(**config).to(device).eval()
x = torch.randn(B, C, T, device=device, dtype=DTYPE)
print(f"\nEntrée : {tuple(x.shape)}  ({x.dtype})")

/home/infres/yrothlin-24/.local/share/mamba/envs/rech202/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
GPU    : NVIDIA L40S
PyTorch: 2.9.1+cu128

Entrée : (8, 64, 6000)  (torch.float32)


## 2. Taille du modèle : paramètres & mémoire des poids

In [2]:
def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    train = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, train

n_total, n_train = count_params(model)
param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buf_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

print(f"Paramètres total       : {n_total/1e6:8.3f} M")
print(f"Paramètres entraînables: {n_train/1e6:8.3f} M")
print(f"Mémoire poids          : {param_bytes/1e6:8.2f} MB")
print(f"Mémoire buffers        : {buf_bytes/1e6:8.2f} MB")
print(f"Empreinte statique     : {(param_bytes+buf_bytes)/1e6:8.2f} MB\n")

# Répartition par sous-module
print(f"{'Module':<18}{'Params (M)':>12}{'% total':>10}")
print("-" * 40)
for name, sub in [
    ("channel_adaptor", model.channel_adaptor),
    ("encoder", model.autoencoder.encoder),
    ("posterior", model.autoencoder.posterior),
    ("decoder", model.autoencoder.decoder),
]:
    p, _ = count_params(sub)
    print(f"{name:<18}{p/1e6:>12.3f}{100*p/n_total:>9.1f}%")

Paramètres total       :    4.830 M
Paramètres entraînables:    4.830 M
Mémoire poids          :    19.32 MB
Mémoire buffers        :     0.27 MB
Empreinte statique     :    19.59 MB

Module              Params (M)   % total
----------------------------------------
channel_adaptor          0.004      0.1%
encoder                  3.027     62.7%
posterior                0.000      0.0%
decoder                  1.799     37.2%


In [3]:
# La méthode summary() intégrée : params par module + shapes de sortie
model.summary(seq_len=T, batch_size=B)

EEGVAE summary  (mode=vq)
Module                Params   Output shape
--------------------------------------------------------
input                          (8, 64, 6000)
ChannelAdaptor         4.29K   (8, 64, 6000)
Encoder                3.03M   (8, 16, 1500)
Posterior                  0   (8, 16, 1500)
Decoder                1.80M   (8, 64, 6000)
--------------------------------------------------------
TOTAL                  4.83M


(tensor([[[-0.5574,  0.2136, -1.2758,  ..., -1.1967, -1.0479,  0.1298],
          [ 0.1617,  1.2404,  1.3499,  ..., -0.1525, -0.0827, -0.1002],
          [ 0.2943, -0.3471,  0.8607,  ..., -1.9138, -0.6638, -0.8407],
          ...,
          [ 0.8684,  2.0955,  1.5067,  ...,  0.9214,  0.6304,  0.6474],
          [ 0.1718,  0.6349, -0.3821,  ...,  0.4947,  0.4751, -0.1786],
          [ 0.5297,  0.1708, -0.8484,  ..., -1.0807, -0.8598,  0.0359]],
 
         [[-0.2873,  1.6430,  1.5372,  ...,  0.4509,  0.2393,  0.0749],
          [-0.2926, -0.1816,  0.0590,  ..., -0.1971,  0.5928,  0.0519],
          [ 0.0164,  0.1090, -1.9659,  ..., -0.8485, -0.4296, -0.1421],
          ...,
          [ 0.5917,  0.5180,  0.7700,  ...,  0.4511, -0.5513, -0.3182],
          [ 0.0160, -0.0189,  0.8817,  ..., -0.1592,  0.1209,  0.7238],
          [-0.0833, -0.4654, -1.0970,  ..., -0.4517, -0.6118, -0.0406]],
 
         [[ 1.0160,  0.2757, -1.8222,  ..., -0.6281, -0.2681, -0.0070],
          [ 0.5560, -0.0732,

## 3. FLOPs / MACs

Deux méthodes croisées. **fvcore** liste les opérateurs non supportés (repère Mamba !) ;
**torch.profiler** compte ce qui s'exécute réellement. Attends-toi à des écarts.

In [4]:
# --- Méthode A : fvcore ---
try:
    from fvcore.nn import FlopCountAnalysis, flop_count_table

    model.eval()
    with torch.no_grad():
        flops = FlopCountAnalysis(model, x)
        flops.unsupported_ops_warnings(False)
        total = flops.total()
    print(flop_count_table(flops, max_depth=2))
    print(f"\nMACs (fvcore)  : {total/1e9:.3f} G   (multiply-add)")
    print(f"FLOPs (~2xMACs): {2*total/1e9:.3f} G")
    print(f"FLOPs / seconde-EEG : {2*total/(B*T/FS)/1e9:.3f} G/s-EEG")
    unsup = dict(flops.unsupported_ops())
    if unsup:
        print("\n⚠️ Opérateurs NON comptés (FLOPs sous-estimés) :")
        for op, cnt in unsup.items():
            print(f"   {op}: {cnt}")
except ImportError:
    print("fvcore non installé  ->  pip install fvcore")
except Exception as e:
    print("fvcore a échoué (probablement à cause des ops Mamba) :", repr(e))

fvcore non installé  ->  pip install fvcore


In [5]:
# --- Méthode B : torch.profiler (ce qui s'exécute vraiment) ---
from torch.profiler import profile, ProfilerActivity

acts = [ProfilerActivity.CPU]
if device == "cuda":
    acts.append(ProfilerActivity.CUDA)

model.eval()
with torch.no_grad():
    model(x)  # warmup
    if device == "cuda":
        torch.cuda.synchronize()
    with profile(activities=acts, with_flops=True) as prof:
        model(x)
        if device == "cuda":
            torch.cuda.synchronize()

total_flops = sum(e.flops for e in prof.key_averages() if e.flops)
print(f"FLOPs (profiler) : {total_flops/1e9:.3f} G\n")
sort_key = "cuda_time_total" if device == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=15))

FLOPs (profiler) : 75.733 G

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  Total KFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                         MambaSplitConv1dScanCombinedFn        24.33%       6.931ms        36.11%      10.287ms     734.812us       3.647ms        28.47%       5.444ms     388.863us            14            --  
                                           aten::matmul         1.48%     422.251us        13.63%       3.883ms     121.32

## 4. Latence & throughput

Warmup + `torch.cuda.synchronize()` obligatoires : les appels CUDA sont asynchrones.

In [6]:
def bench(fn, n_warmup=10, n_iter=50):
    """Retourne (mean, std, p50, p95) en millisecondes."""
    for _ in range(n_warmup):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()
    ts = []
    for _ in range(n_iter):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
    ts = np.asarray(ts) * 1e3
    return ts.mean(), ts.std(), np.percentile(ts, 50), np.percentile(ts, 95)

model.eval()
print(f"{'Op':<14}{'mean±std (ms)':>18}{'p50':>8}{'p95':>8}"
      f"{'samp/s':>10}{'s-EEG/s':>10}")
print("-" * 68)
with torch.no_grad():
    for name, fn in [
        ("forward",     lambda: model(x)),
        ("encode",      lambda: model.encode(x)),
        ("reconstruct", lambda: model.reconstruct(x)),
    ]:
        m, s, p50, p95 = bench(fn)
        thr = B / (m / 1e3)
        eeg_s = (B * T) / (m / 1e3) / FS
        print(f"{name:<14}{m:8.2f} ± {s:5.2f}{p50:>8.2f}{p95:>8.2f}"
              f"{thr:>10.1f}{eeg_s:>10.1f}")

Op                 mean±std (ms)     p50     p95    samp/s   s-EEG/s
--------------------------------------------------------------------
forward          15.16 ±  0.06   15.15   15.30     527.6   12661.9
encode            8.50 ±  0.03    8.49    8.54     941.3   22590.0
reconstruct      15.71 ±  0.78   15.21   17.07     509.1   12218.6


In [7]:
# Latence d'un pas d'entraînement : forward + backward
def train_step():
    out = model(x)
    dec = out[0] if isinstance(out, tuple) else out
    loss = dec.float().pow(2).mean()
    loss.backward()
    model.zero_grad(set_to_none=True)

model.train()
m, s, p50, p95 = bench(train_step, n_warmup=5, n_iter=20)
print(f"Train step (fwd+bwd) : {m:.2f} ± {s:.2f} ms  (p95 {p95:.2f})")
print(f"Throughput train     : {B/(m/1e3):.1f} samples/s")
model.eval()

Train step (fwd+bwd) : 57.20 ± 0.52 ms  (p95 57.83)
Throughput train     : 139.9 samples/s


EEGVAE(
  (channel_adaptor): ChannelAdaptor(
    (model): Sequential(
      (0): Conv1d(64, 32, kernel_size=(1,), stride=(1,), bias=False)
      (1): Norm(
        (norm): GroupNorm(8, 32, eps=1e-05, affine=True)
      )
      (2): ReLU(inplace=True)
      (3): Conv1d(32, 64, kernel_size=(1,), stride=(1,), bias=False)
      (4): Norm(
        (norm): GroupNorm(8, 64, eps=1e-05, affine=True)
      )
      (5): ReLU(inplace=True)
    )
  )
  (autoencoder): AutoEncoder(
    (encoder): Encoder(
      (down): ModuleList(
        (0): Module(
          (block): ModuleList(
            (0-1): 2 x MambaBlock(
              (conv_block): Sequential(
                (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
                (1): Norm(
                  (norm): GroupNorm(8, 64, eps=1e-05, affine=True)
                )
                (2): SiLU(inplace=True)
                (3): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
                (4): Norm(
          

## 5. Mémoire GPU (pic réel)

In [8]:
if device == "cuda":
    # Inférence
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model.eval()
    with torch.no_grad():
        model(x)
    torch.cuda.synchronize()
    infer_peak = torch.cuda.max_memory_allocated() / 1e6

    # Entraînement (forward + backward)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model.train()
    out = model(x)
    dec = out[0] if isinstance(out, tuple) else out
    loss = dec.float().pow(2).mean()
    loss.backward()
    torch.cuda.synchronize()
    train_peak = torch.cuda.max_memory_allocated() / 1e6
    model.zero_grad(set_to_none=True)
    model.eval()

    print(f"Peak VRAM inférence     : {infer_peak:8.1f} MB")
    print(f"Peak VRAM entraînement  : {train_peak:8.1f} MB")
    print(f"Surcoût activations+grad: {train_peak-infer_peak:8.1f} MB")
else:
    print("Mesure VRAM indisponible sur CPU.")

Peak VRAM inférence     :    295.7 MB
Peak VRAM entraînement  :   2931.5 MB
Surcoût activations+grad:   2635.8 MB


## 6. Scaling : latence & VRAM vs (batch, longueur)

La courbe la plus utile en pratique : trouve le batch/seq_len max avant l'OOM.

In [9]:
model.eval()
print(f"{'B':>4}{'T':>7}{'latence (ms)':>16}{'VRAM (MB)':>12}")
print("-" * 40)
for Ts in [1000, 3000, 6000, 12000]:
    for Bs in [1, 4, 16]:
        try:
            xb = torch.randn(Bs, C, Ts, device=device, dtype=DTYPE)
            if device == "cuda":
                torch.cuda.reset_peak_memory_stats()
            with torch.no_grad():
                m, *_ = bench(lambda: model(xb), n_warmup=3, n_iter=10)
            vram = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else 0
            print(f"{Bs:>4}{Ts:>7}{m:>16.2f}{vram:>12.0f}")
            del xb
        except RuntimeError as e:
            msg = "OOM" if "out of memory" in str(e).lower() else "ERR"
            print(f"{Bs:>4}{Ts:>7}{msg:>16}")
            if device == "cuda":
                torch.cuda.empty_cache()

   B      T    latence (ms)   VRAM (MB)
----------------------------------------
   1   1000           14.32          88
   4   1000           14.58         104
  16   1000           14.53         167
   1   3000           14.63          99
   4   3000           14.57         145
  16   3000           15.43         328
   1   6000           14.47         114
   4   6000           14.83         205
  16   6000           31.10         574
   1  12000           14.38         145
   4  12000           15.83         328
  16  12000           75.02        1053


## 7. (Optionnel) Mixed precision — autocast

Compare la latence FP32 vs FP16/BF16 si tu entraînes en AMP.

In [10]:
if device == "cuda":
    x32 = torch.randn(B, C, T, device=device, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        m_fp32, *_ = bench(lambda: model(x32))
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            m_bf16, *_ = bench(lambda: model(x32))
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            m_fp16, *_ = bench(lambda: model(x32))
    print(f"forward FP32          : {m_fp32:7.2f} ms  (x1.00)")
    print(f"forward autocast BF16 : {m_bf16:7.2f} ms  (x{m_fp32/m_bf16:.2f})")
    print(f"forward autocast FP16 : {m_fp16:7.2f} ms  (x{m_fp32/m_fp16:.2f})")
else:
    print("Autocast pertinent surtout sur GPU.")

forward FP32          :   15.52 ms  (x1.00)
forward autocast BF16 :   16.61 ms  (x0.93)
forward autocast FP16 :   16.04 ms  (x0.97)


## 8. Discriminateur (si utilisé en entraînement adversarial)

Même méthodologie appliquée au `Discriminator`.

In [11]:
from model.discriminator import Discriminator

disc = Discriminator(in_channels=C, ndf=64, n_layers=3).to(device).eval()
dn_total, dn_train = count_params(disc)
print(f"Discriminateur : {dn_total/1e6:.3f} M params")

with torch.no_grad():
    m, s, *_ = bench(lambda: disc(x))
print(f"forward disc   : {m:.2f} ± {s:.2f} ms  ({B/(m/1e3):.1f} samples/s)")
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        disc(x)
    torch.cuda.synchronize()
    print(f"Peak VRAM disc : {torch.cuda.max_memory_allocated()/1e6:.1f} MB")

Discriminateur : 0.708 M params
forward disc   : 0.34 ± 0.01 ms  (23282.2 samples/s)
Peak VRAM disc : 137.3 MB
